# FairQueue Simulator — results review

Walkthrough of the processed outputs: waiting-list pressure, fairness risk, operational pressure, the final priority score, and the five-strategy simulation.

Run top-to-bottom after the pipeline (`src/02`–`src/07`) has produced the parquet files.

In [ ]:
import pandas as pd, plotly.express as px
from pathlib import Path
P = Path.cwd().parent / 'data' / 'processed' if (Path.cwd().name=='notebooks') else Path('data/processed')
scores = pd.read_parquet(P/'priority_scores.parquet')
rtt = pd.read_parquet(P/'rtt_provider_specialty_month.parquet')
strat = pd.read_parquet(P/'strategy_comparison.parquet')
latest = scores.month.max()
print('rows:', len(scores), '| months:', scores.month.nunique(), '| latest:', latest)

## 1. National waiting-list trend

In [ ]:
g = rtt.groupby('month').agg(incomplete=('incomplete_total','sum'),
    b18=('breach_18w_count','sum'), b52=('breach_52w_count','sum'))
g['pct_within_18w'] = (100*(1-g.b18/g.incomplete)).round(1)
g

## 2. Top priority areas (full fairness-aware model)

In [ ]:
cols=['provider_name','treatment_function_name','waiting_pressure_score',
      'fairness_risk_score','operational_pressure_score','priority_score_100','priority_level']
scores[scores.month==latest].nlargest(15,'priority_score_100')[cols].round(2)

## 3. Does fairness change prioritisation?
Compare the top-20 of each strategy with the waiting-time-only baseline.

In [ ]:
k=20
g=strat[strat.month==latest]
base=set(g.nsmallest(k,'rank_waiting_time_only')[['provider_code','treatment_function_code']].apply(tuple,axis=1))
for s in ['operational','cancellation_aware','deprivation_weighted','full_fairness_aware']:
    cur=set(g.nsmallest(k,f'rank_{s}')[['provider_code','treatment_function_code']].apply(tuple,axis=1))
    print(f'{s:22s} top-20 overlap with waiting-only: {len(base&cur)/k:.0%}')

## 4. Biggest movers under the fairness-aware model

In [ ]:
m=strat[strat.month==latest].nlargest(12,'shift_full_fairness_aware')
m[['provider_name','treatment_function_name','rank_waiting_time_only',
   'rank_full_fairness_aware','shift_full_fairness_aware']]

## 5. Score distribution by priority component

In [ ]:
px.scatter(scores[scores.month==latest], x='waiting_pressure_score', y='fairness_risk_score',
    size='priority_score_100', color='operational_pressure_score', hover_name='provider_name',
    title='Waiting vs fairness (size = priority, colour = operational)')

## Notes for the paper
- Low top-20 overlap (~25% for the full model, 0% deprivation-weighted) = fairness-aware scoring materially changes which areas are flagged.
- See `outputs/metrics/` for fairness/operational/cancellation lifts and the sensitivity analysis.
- All scores are transparent weighted sums of min-max-normalised public NHS metrics — no black box.